In [1]:
import os
import copy
import pandas as pd
from scipy.stats import fisher_exact

infile = '../../crc_analysis_12177/data/merged_hgt.csv'
infile2 = '../../crc_analysis_12177/data/metadata.tsv'
db_idir = '../../HGT_demo_file/DB.VFDB.anno'
groupid = 'phenotype'
outdir = '.'
fr_size = 1000

df = pd.read_csv(infile, header=0, index_col=None)
df.rename(columns={'receptor':'recipient'}, inplace=True)
metadata = pd.read_csv(infile2, header=0, index_col=0, sep='\t')
if len(metadata[groupid].unique()) != 2:
    print('Error: the column {} does not have exact 2 level.'.format(groupid))
    exit(1)
hgt_slist = list(set(df['sample']))
g_slist = list(metadata.index)
valid = True
for s in list(hgt_slist):
    if s not in g_slist:
        print('Error: group information of sample {} in HGT event dose NOT exist.'.format(s))
        valid = False
if not valid:
    exit(2)

In [2]:
def overlap(range1, range2):
    if range1[0] > range2[1] or range1[1] < range2[0]:
        return False
    else:
        return True

# styp = recipient or donor
def search_event(scaffold, db, range):
    scaffold_df = db[db['Chr'] == scaffold]
    valid_idx = []
    for idx in scaffold_df.index:
        if overlap(range, [scaffold_df.loc[idx, 'Start'], scaffold_df.loc[idx, 'End']]):
            valid_idx.append(idx)
    valid_df = scaffold_df.loc[valid_idx, ]
    return valid_df
    

def search_row(idx, df, db_dir, fr_size):
    tmp = '{}.VFDB.tsv'
    row = df.loc[idx, ]
    # for recipient
    recipient = row['recipient']
    chrom = recipient.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    db = pd.read_csv(ifile, header=0, index_col=None, sep='\t')
    recipient_range = [max(0, row['insert_locus']-fr_size), row['insert_locus']+fr_size]
    recipient_df = search_event(recipient, db, recipient_range)
    # for donor
    donor = row['donor']
    range = [max(0, row['delete_start'] - fr_size), row['delete_end'] + fr_size]
    chrom = donor.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    db = pd.read_csv(ifile, header=0, index_col=None, sep='\t')
    donor_df = search_event(donor, db, range)
    return recipient_df, donor_df 

def enrichment(metadata, result_anno, groupid, outdir):
    pheno_set = list(set(metadata[groupid]))
    g1 = pheno_set[0]
    g2 = pheno_set[1]
    vf_df = result_anno[(result_anno['recipient_VF_n']>0) | (result_anno['donor_VF_n']>0)]
    cate_df = pd.DataFrame(columns=[g1, g2])
    for idx in vf_df.index:
        recipient_VF_category = vf_df.loc[idx, 'recipient_VF_category'].split(';')
        donor_VF_category = vf_df.loc[idx, 'donor_VF_category'].split(';')
        all_cates = recipient_VF_category+donor_VF_category
        for cate in all_cates:
            if cate == 'NA':
                continue
            if cate not in cate_df.index:
                cate_df.loc[cate, g1] = 0
                cate_df.loc[cate, g2] = 0
            cate_df.loc[cate, metadata.loc[vf_df.loc[idx, 'sample'], groupid]] += 1
    pvalue_reformat = pd.DataFrame(columns=['group1', 'group2', 'category', 'g1_in_category', 'g1_total', 'g2_in_category', 'g2_total', 'pvalue', 'odds_ratio'])
    g1_total = cate_df[g1].sum()
    g2_total = cate_df[g2].sum()
    for cate in cate_df.index:
        a = cate_df.loc[cate, g1]
        b = cate_df.loc[cate, g2]
        c = g1_total - a
        d = g2_total - b
        if a+b == 0 or c+d == 0 or a+c == 0 or b+d == 0:
            pvalue_reformat.loc[cate, ] = [g1, g2, cate, a, g1_total, b, g2_total, 'NA', 'NA']
            continue
        oddsratio, pvalue = fisher_exact([[a, b], [c, d]])
        pvalue_reformat.loc[cate, ] = [g1, g2, cate, a, g1_total, b, g2_total, pvalue, oddsratio]
    return pvalue_reformat
   

In [3]:
result_anno = pd.DataFrame(columns=['id', 'sample', 'recipient_VF_n', 'recipient_VF_category', 'recipient_VF_list', 'donor_VF_n', 'donor_VF_category', 'donor_VF_list', 'recipient', 'insert_locus', 'donor', 'delete_start', 'delete_end', 'reverse_flag'])
MGE_result = pd.DataFrame()
for idx in df.index:
    recipient_df, donor_df = search_row(idx, df, db_idir, fr_size)
    id = 'HGT_c{}'.format(idx+1)
    sample = df.loc[idx, 'sample']
    recipient_MGE_n = recipient_df.shape[0]
    recipient_MGE_category = ';'.join(recipient_df['Category'])
    recipient_MGE_list = ';'.join(recipient_df['Name'])
    if recipient_MGE_n == 0:
        recipient_MGE_list = 'NA'
        recipient_MGE_category = 'NA'
    donor_MGE_n = donor_df.shape[0]
    donor_MGE_category = ';'.join(recipient_df['Category'])
    donor_MGE_list = ';'.join(recipient_df['Name'])
    if donor_MGE_n == 0:
        donor_MGE_list = 'NA'
        donor_MGE_category = 'NA'
    recipient = df.loc[idx, 'recipient']
    insert_locus = df.loc[idx, 'insert_locus']
    donor = df.loc[idx, 'donor']
    delete_start = df.loc[idx, 'delete_start']
    delete_end = df.loc[idx, 'delete_end']
    reverse_flag = df.loc[idx, 'reverse_flag']
    result_anno.loc[len(result_anno), ] = [id, sample, recipient_MGE_n, recipient_MGE_category, recipient_MGE_list, donor_MGE_n, donor_MGE_category, donor_MGE_list, recipient, insert_locus, donor, delete_start, delete_end, reverse_flag]
    #result_anno.iloc[len(result_anno), ] = [id, sample, recipient_HGTC_n, recipient_HGTC_list, donor_HGTC_n, donor_HGTC_list, recipient, insert_locus, donor, delete_start, delete_end, reverse_flag]
    merge_df = pd.concat([recipient_df, donor_df], ignore_index=True)
    if MGE_result.empty:
        MGE_result = copy.deepcopy(merge_df)
    else:
        MGE_result = pd.concat([MGE_result, merge_df], ignore_index=True)
MGE_result.drop_duplicates(inplace=True)
MGE_result.to_csv(os.path.join(outdir, 'output.VF_comparison.VF.tsv'), index=False, sep='\t')
result_anno.to_csv(os.path.join(outdir, 'output.VF_comparison.annotated.tsv'), index=False, sep='\t')
pvalue_df = enrichment(metadata, result_anno, groupid, outdir)
pvalue_df.to_csv(os.path.join(outdir, 'output.VF_comparison.pvalue.tsv'), index=False, sep='\t')


In [4]:
fisher_exact([[9, 0], [9, 9]])

SignificanceResult(statistic=inf, pvalue=0.011583748059720601)

In [5]:
fisher_exact([[0, 9], [9, 9]])

SignificanceResult(statistic=0.0, pvalue=0.0115837480597206)

In [6]:
fisher_exact([[5, 9], [0, 9]])

SignificanceResult(statistic=inf, pvalue=0.11566465571042228)

In [7]:
fisher_exact([[1, 9], [9, 0]])

SignificanceResult(statistic=0.0, pvalue=0.00011907597046915933)

In [8]:
fisher_exact([[0, 0], [9, 9]])

SignificanceResult(statistic=nan, pvalue=1.0)

In [9]:
fisher_exact([[0, 9], [0, 90]])

SignificanceResult(statistic=nan, pvalue=1.0)

In [10]:
fisher_exact([[1, 9], [0, 0]])

SignificanceResult(statistic=nan, pvalue=1.0)

In [11]:
fisher_exact([[7, 0], [9, 0]])

SignificanceResult(statistic=nan, pvalue=1.0)